Explainable Suicide Risk Detection — Transformer Pipeline
























In [ ]:
from huggingface_hub import login
login()
# Needed for the gated "AIMH/mental-roberta-large" backbone this notebook now defaults
# to -- accept its license at https://huggingface.co/AIMH/mental-roberta-large (while
# logged in to HuggingFace) BEFORE running this cell, then paste a token that has access
# when prompted. If you skip this, the CV cell detects it and falls back to the
# base-size domain model automatically, so nothing breaks -- you'll just miss the
# larger model's extra capacity until you come back and do this.

In [ ]:
!pip install -q transformers scikit-learn openpyxl accelerate

In [ ]:
import re, ast, html, unicodedata, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU, then re-run.")

## Configuration
Adjust these, then Runtime → Run all.

In [ ]:
MODEL_NAME = "mental/mental-roberta-base"  # pinned back explicitly -- three straight 403s on
                                    # AIMH/mental-roberta-large (gate still not accepted). Rather than
                                    # have every future run silently depend on that clearing, this is now
                                    # an explicit, deterministic choice: the proven base-size domain
                                    # model. If you get large-model access later, swapping this one line
                                    # back is all it takes -- everything else (pooler, both risk heads,
                                    # factors head) already resizes itself automatically.
MODEL_NAME_FALLBACK = "roberta-base"  # plain, ungated -- MODEL_NAME above is itself now the proven
                                    # base-size domain model, so this only matters in the unlikely case
                                    # even THAT can't load (e.g. access revoked, HF outage).
MAX_LENGTH = 512                   # RoBERTa's max; long posts (>~380 words) get truncated -- see note in the CV cell
N_FOLDS = 8                         # back down from a brief 12 -- that didn't show a clear benefit over
                                    # 8 last round, and this round already adds ~30-60 min for the new
                                    # LLM cell above (see markdown) on top of the fold loop itself. 8
                                    # keeps total runtime closer to ~90-100 min all-in instead of 2+
                                    # hours, which matters more than a marginal ensemble-size lever right
                                    # now given how little time is left.
EPOCHS = 10                         # REVERTED from 13 -- that run scored 0.6681, which falls inside the
                                    # noise band already established by identical code at EPOCHS=10
                                    # (0.6688-0.6793 across two runs). No clear signal either way, so
                                    # defaulting back to the longer-validated setting rather than
                                    # carrying forward a change nothing actually confirmed helps.
BATCH_SIZE = 8                      # restored -- 4 was only ever a memory accommodation for the large
                                     # model, which is no longer what is being loaded.
FREEZE_BOTTOM_FRACTION = 0.1        # REVERTED from 0.0 -- the sweep is complete and conclusive:
                                     #   0.50 -> real composite 0.6637
                                     #   0.25 -> real composite 0.6645
                                     #   0.10 -> real composite 0.6793  <- best, by a real margin
                                     #   0.00 -> real composite 0.6652  (both subtasks down together,
                                     #                                   not just noise on one)
                                     # A clean inverted U: too frozen underfits, fully unfrozen overfits
                                     # a 1635-row dataset with this much trainable capacity. 0.1 is the
                                     # confirmed optimum on this axis -- not touching it again without a
                                     # specific new reason to revisit.
FREEZE_EMBEDDINGS = True            # reverted together with FREEZE_BOTTOM_FRACTION above -- these two
                                     # move as a pair (see the model code's own comment on why).
BACKBONE_LR = 2e-5                  # paired with the freeze settings above.
HEAD_LR = 1e-4
SEED = 42

USE_EMA = False                    # was on, reverted: decay=0.999 needs tens of thousands of steps to
EMA_DECAY = 0.999                  # converge -- this run has only ~800, so the EMA shadow stayed ~44%
                                    # weighted toward the near-random INITIAL state through the whole run.
                                    # Confirmed by direct calculation, not just suspicion -- see chat.
USE_FGM = False                    # was on, reverted: epsilon=1.0 is very likely too large relative to
FGM_EPSILON = 1.0                  # RoBERTa's actual embedding norm scale, injecting noise into the shared
                                    # encoder that ALL THREE heads sit on top of -- consistent with both
                                    # Subtask 1 and Subtask 2 dropping together in the last real submission.
                                    # Both flags are left here (rather than deleted) so you can experiment
                                    # with them later, once this round's changes are confirmed working --
                                    # e.g. EMA_DECAY≈0.99 would be far better calibrated for a run this short.

USE_MEAN_POOLING = True            # pool CLS + attention-mask-weighted mean of all tokens (not CLS
                                    # alone) through a small MLP before the risk/factors heads. RoBERTa's
                                    # <s> token has no NSP-style pretraining objective the way BERT's
                                    # [CLS] does, so mean-pooling over real tokens typically carries more
                                    # usable signal for downstream classification -- well-established in
                                    # the sentence-embedding literature (e.g. Sentence-BERT). Evidence
                                    # extraction is unaffected -- it already reads every token's own
                                    # hidden state, not a pooled summary.

RARE_FACTOR_THRESHOLD = 50          # factor categories with fewer than this many positive examples
                                     # (within a fold's training portion) get their rows oversampled --
                                     # covers the categories that predicted ~0/378 on the real leaderboard
                                     # submission (sexual orientation related issues, poor school
                                     # performance, cognitive deficits), plus a couple more nearby.
OVERSAMPLE_FACTOR = 3                # rare-category rows appear 3x in training; everything else 1x.

MIN_SUPPORT_FOR_THRESHOLD_TUNING = 15  # a factor category needs at least this many positive examples
                                        # in the POOLED out-of-fold set before we trust a searched
                                        # decision threshold for it; below that we keep 0.5. Replaces
                                        # the old per-fold dev-split tuning (min_support=3 on ~150-row
                                        # slices), which is what was quietly overfitting: e.g. "poor
                                        # school performance" (true prevalence ~1%) ended up with a
                                        # threshold so low it fired on 27% of validation rows (F1 0.03) --
                                        # picked from a handful of dev-split positives, not a signal that
                                        # actually held up. Pooling OOF across all 1635 training rows
                                        # (instead of ~245 per fold) gives every category far more
                                        # examples to tune against -- see the CV cell for how this is used.

FACTOR_THRESHOLD_FBETA = 0.7        # NOW APPLIED CONDITIONALLY, not to every category -- see
                                     # FACTOR_OVERPREDICT_RATIO below. Applying this uniformly to all
                                     # 24 categories (previous round) measurably REGRESSED the real
                                     # leaderboard's factors score (0.4184 -> 0.4085) despite improving
                                     # the pooled-OOF estimate (0.3968 -> 0.4301) -- classic CV/
                                     # leaderboard divergence. The mechanism was visible in the actual
                                     # per-category deltas: categories that WERE over-predicting got
                                     # meaningfully better (sense of responsibility F1 +0.056, physical
                                     # health +0.086), but already-well-calibrated ones got pushed
                                     # toward needless under-prediction and got WORSE (hopelessness,
                                     # ~46% prevalence, F1 -0.029; emotion dysregulation -0.007) --
                                     # since macro F1 weighs all 24 categories equally, that's enough to
                                     # flip the net real-world effect negative even though the
                                     # genuinely-helped categories individually improved more. beta=1.0
                                     # reproduces plain F1 for a category regardless of this flag.
FACTOR_OVERPREDICT_RATIO = 2.0      # a category only gets the FACTOR_THRESHOLD_FBETA precision bias if
                                     # its predicted rate AT ITS OWN PLAIN-F1-OPTIMAL THRESHOLD already
                                     # exceeds this many times its true prevalence -- i.e. only
                                     # categories the model itself is demonstrably over-predicting once
                                     # tuned. (Checking at a fixed 0.5 instead of each category's own
                                     # operating point caught only 1/24 categories in practice and
                                     # missed clear over-predictors -- see the CV cell.) Categories that
                                     # pass MIN_SUPPORT_FOR_THRESHOLD_TUNING but aren't over-predicting
                                     # (like hopelessness) use plain F1 regardless of FACTOR_THRESHOLD_FBETA.

TRAIN_PATH = "train.xlsx"           # upload these two files to the Colab file browser first,
LEADERBOARD_PATH = "leaderboard.xlsx"  # or mount Drive and point these paths there
TEAM_NAME = "4aumArivu"        # <-- set this; used for the output filename

torch.manual_seed(SEED)
np.random.seed(SEED)

## Data loading & cleaning
Same cleaning as the classical baseline: HTML-unescape posts, normalize risk-label casing, de-duplicate factor tags, drop `none`/`n/a` placeholder evidence spans.

In [ ]:
RISK_LEVELS = ["Indicator", "Ideation", "Behavior", "Attempt"]
RISK_TO_IDX = {r: i for i, r in enumerate(RISK_LEVELS)}
FACTOR_TAXONOMY = [
    "mental health issues", "physical health/characteristic", "substance use",
    "hopelessness", "emotion dysregulation", "low self-esteem",
    "poor school performance", "low socio-economic status", "interpersonal violence",
    "prior self-harm or suicidal thought/attempt", "poor social support",
    "interpersonal difficulty", "dysfunctional family", "exposure to others' suicide",
    "stressful life event", "traumatic experience", "cognitive deficits",
    "suicide means (with access)", "sexual orientation related issues",
    "social support", "coping strategy", "psychological capital",
    "sense of responsibility", "meaning in life",
]
PLACEHOLDER = {"none", "n/a", "na", "nil", "-", ""}
RISK_CANON = {r.lower(): r for r in RISK_LEVELS}

def clean_post(s):
    return html.unescape(str(s))

def parse_row(row):
    post = clean_post(row["post"])
    risk = RISK_CANON.get(str(row["suicide risk"]).strip().lower(), row["suicide risk"])
    try:
        factors = sorted(set(ast.literal_eval(row["factors"])))
    except Exception:
        factors = []
    spans = []
    ev_raw = row["evidence for suicide risk level"]
    if pd.notna(ev_raw):
        for s in str(ev_raw).split(";"):
            s = s.strip()
            if s and s.lower() not in PLACEHOLDER:
                spans.append(s)
    return post, risk, factors, spans

def load(path):
    df = pd.read_excel(path)
    has_labels = "suicide risk" in df.columns
    posts, risks, factors, spans_list = [], [], [], []
    for _, row in df.iterrows():
        if has_labels:
            post, risk, factors_row, spans = parse_row(row)
        else:
            post, risk, factors_row, spans = clean_post(row["post"]), None, None, None
        posts.append(post); risks.append(risk); factors.append(factors_row); spans_list.append(spans)
    df["post_clean"] = posts
    if has_labels:
        df["risk_clean"] = risks
        df["factors_clean"] = factors
        df["evidence_clean"] = spans_list
    return df

train_df = load(TRAIN_PATH)
leaderboard_df = load(LEADERBOARD_PATH)
print(f"train: {len(train_df)} rows, {train_df['anon_user_id'].nunique()} users")
print(f"leaderboard: {len(leaderboard_df)} rows, {leaderboard_df['anon_user_id'].nunique()} users")
print("user overlap between the two:", len(set(train_df.anon_user_id) & set(leaderboard_df.anon_user_id)), "(should be 0)")

## Official scoring functions
Implemented literally from the competition rules and self-tested against the worked example, the 3× length cap, and the one-to-one matching constraint (see the standalone `scoring.py` from the classical baseline for the full test suite).

In [ ]:
def _norm(s):
    s = html.unescape(str(s))
    s = unicodedata.normalize("NFKC", s)
    s = (s.replace("\u2019","'").replace("\u2018","'").replace("\u201c",'"').replace("\u201d",'"')
           .replace("\u2013","-").replace("\u2014","-"))
    return re.sub(r"\s+", " ", s).strip().lower()

def _tok_len(s):
    return len(re.findall(r"\w+", s))

def weighted_f1_risk(y_true, y_pred, labels=RISK_LEVELS):
    y_true_n = [str(y).strip().lower() for y in y_true]
    y_pred_n = [str(y).strip().lower() for y in y_pred]
    labels_n = [l.lower() for l in labels]
    return f1_score(y_true_n, y_pred_n, labels=labels_n, average="weighted", zero_division=0)

def _compatible(pred_norm, gold_norm):
    if pred_norm == "" or gold_norm == "":
        return False
    if not (pred_norm in gold_norm or gold_norm in pred_norm):
        return False
    return _tok_len(pred_norm) <= 3 * _tok_len(gold_norm)

def _max_bipartite_matching(adj, n_left, n_right):
    match_right = [-1] * n_right
    def try_assign(i, seen):
        for j in adj[i]:
            if j in seen: continue
            seen.add(j)
            if match_right[j] == -1 or try_assign(match_right[j], seen):
                match_right[j] = i
                return True
        return False
    count = 0
    for i in range(n_left):
        if try_assign(i, set()):
            count += 1
    return count

def phrase_f1_single(pred_spans, gold_spans):
    preds = [_norm(p) for p in pred_spans if _norm(p)]
    golds = [_norm(g) for g in gold_spans if _norm(g)]
    if not preds and not golds:
        return 1.0, 1.0, 1.0
    adj = [{j for j, g in enumerate(golds) if _compatible(p, g)} for p in preds]
    tp = _max_bipartite_matching(adj, len(preds), len(golds))
    precision = tp / len(preds) if preds else 0.0
    recall = tp / len(golds) if golds else 0.0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0.0
    return precision, recall, f1

def phrase_f1_corpus(pred_spans_list, gold_spans_list):
    f1s = [phrase_f1_single(p, g)[2] for p, g in zip(pred_spans_list, gold_spans_list)]
    return sum(f1s) / len(f1s) if f1s else 0.0

def macro_f1_factors(true_factor_lists, pred_factor_lists, taxonomy=FACTOR_TAXONOMY):
    mlb = MultiLabelBinarizer(classes=taxonomy)
    y_true = mlb.fit_transform([set(f) for f in true_factor_lists])
    y_pred = mlb.transform([set(f) for f in pred_factor_lists])
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

def composite_score(risk_f1, evidence_f1, factors_f1):
    return 0.4*risk_f1 + 0.3*evidence_f1 + 0.3*factors_f1

##Model architecture
Shared encoder with three heads, `mental/mental-roberta-base` — continued-pretrained on mental-health Reddit posts (Ji et al. 2022), so the starting vocabulary/representations are already closer to this domain (r/SuicideWatch) before any fine-tuning happens. A large (24-layer) variant of the same family exists (`AIMH/mental-roberta-large`) and would need zero architecture changes to try — `hidden_size` is read dynamically off the loaded encoder everywhere below, never hardcoded — but three straight gate rejections mean it isn't what's actually running; see config for the current pin.

Pooling: risk and factors now read from `mean_pool(seq, attention_mask)` concatenated with the `<s>`/CLS token, projected through a small MLP (`Linear → GELU → LayerNorm → Dropout`) before the final per-task layers, rather than the bare CLS token straight into a single `Linear`. RoBERTa's `<s>` token has no NSP-style objective pushing it to summarize the sequence the way BERT's `[CLS]` does, so a mean over real tokens plus a bit of extra head capacity is a well-established, low-risk change (`USE_MEAN_POOLING` in config). Evidence extraction is untouched by this — it already reads every token's own hidden state, no pooling involved.

**Risk and factors now get separate poolers** (they shared one in the previous round). The real submission's error analysis showed several factor categories with an already-well-calibrated predicted rate that still scored low F1 — a discrimination problem, not a threshold problem, consistent with a 24-way multi-label head and a 4-way ordinal head fighting over one shared projection's gradients. Splitting them is cheap (~1M extra params) and gives factors — still clearly the lagging task — its own capacity and gradient path.

**This round, an aggressive push** (deadline-driven, not the usual one-lever-at-a-time pace): the training loss now weights factors 0.4 / evidence 0.2 (was 0.3 / 0.3) -- more gradient budget for the demonstrably hardest task, pulled from evidence rather than risk since evidence has the most headroom to spare. Separately, a classical TF-IDF + per-category logistic-regression model (`class_weight="balanced"`, the same approach the very first baseline in this project used) is fit fresh per fold on the same oversampled training posts, and blended with the neural factor probabilities via a pooled-OOF-searched weight -- a genuinely different model family, not another hyperparameter on the same transformer, on the theory that lexical/n-gram signal may generalize differently than a fine-tuned encoder for the categories with the fewest training examples. Both changes are checked against the pooled-OOF numbers in the CV cell, same as everything else, but bundled together this round given the time constraint -- if the real submission is mixed, these two are the first things to isolate from each other in a follow-up.

Risk level gets a standard classifier *and* a CORAL ordinal head (the two are ensembled at inference by averaging class probabilities) since risk is genuinely ordinal (Indicator<Ideation<Behavior<Attempt) — this dual-head idea is validated for this exact competition lineage in arXiv:2510.20085. Factors uses **alpha-weighted** focal loss (alpha=1-prevalence, clipped to [0.15, 0.85] — a synthetic 24-category test matching this dataset's real prevalence spread showed the tighter [0.05, 0.95] clip overcorrects into broad over-prediction; [0.15, 0.85] recovered the rare-category benefit without it) plus **per-category decision thresholds**, now tuned ONCE on pooled out-of-fold predictions across the full train set instead of per-fold on a small dev split — see the CV cell for why (short version: the old per-fold approach was overfitting rare-category thresholds to single-digit example counts). Evidence gets the same pooled-OOF tuning treatment this round, for its (threshold, merge_gap) decode pair — previously fixed at untuned defaults.

**EMA and FGM are implemented but OFF by default** (see the config cell) after a real submission with both enabled scored worse across *both* subtasks than the version without them:
- **EMA** at decay=0.999 needs tens of thousands of steps to converge; this run has roughly 800 total. Direct calculation: `0.999^800 ≈ 0.44`, meaning the EMA shadow used for eval was still ~44% weighted toward the near-random initial state even at the end of training. Wrong decay for a run this short, not a broken idea in general — `EMA_DECAY≈0.99` would be far better calibrated, if you want to re-enable it later.
- **FGM** perturbs the embedding layer's gradient direction (which is why `freeze_embeddings` below is tied to `USE_FGM` — a frozen embedding layer has no gradient to attack, so FGM against it silently does nothing). `epsilon=1.0` is very likely too large relative to RoBERTa's actual embedding norm scale, injecting noise into the shared encoder that all three heads sit on top of.

Both are left in the code (rather than removed) since they're reasonable techniques *in general* — they were just miscalibrated for this specific short training regime, and the honest move after a real regression is to revert to the last validated configuration, not keep guessing.

**Also skipped, unchanged from before:** SWA (redundant with EMA), pseudo-labeling (training on the model's own still-imperfect predictions would compound whatever it's currently getting wrong), heavy feature-importance analysis (the confusion matrix below covers the practical need at much lower cost). **New this round, deliberately not attempted:** augmenting the oversampled rare-category duplicates (rather than repeating them verbatim) — real potential upside, but doing it safely means never touching characters inside a gold evidence span (or the evidence label alignment silently breaks for that row), and that couldn't be verified without a real training run. Worth trying as its own isolated experiment once the changes here are confirmed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaModel

NUM_RISK_CLASSES = 4
NUM_FACTORS = 24


class CoralHead(nn.Module):
    """Ordinal head: K-1 cumulative-probability thresholds P(y>k), shared weight
    vector + strictly-decreasing biases (enforced via softplus deltas), which is
    what guarantees P(y>0) >= P(y>1) >= P(y>2) -- i.e. rank-consistent predictions."""

    def __init__(self, hidden_size, num_classes=NUM_RISK_CLASSES):
        super().__init__()
        self.num_classes = num_classes
        self.shared = nn.Linear(hidden_size, 1, bias=False)
        self.bias_deltas = nn.Parameter(torch.zeros(num_classes - 1))

    def ordered_biases(self):
        b0 = self.bias_deltas[0:1]
        deltas = F.softplus(self.bias_deltas[1:])
        return torch.cat([b0, b0 - torch.cumsum(deltas, dim=0)])

    def forward(self, x):
        return self.shared(x) + self.ordered_biases().unsqueeze(0)  # (batch, num_classes-1)

    def loss(self, logits, labels):
        k = torch.arange(self.num_classes - 1, device=labels.device).unsqueeze(0)
        targets = (labels.unsqueeze(1) > k).float()
        return F.binary_cross_entropy_with_logits(logits, targets)

    def predict(self, logits):
        return (torch.sigmoid(logits) > 0.5).sum(dim=1)


def focal_bce(logits, targets, gamma=2.0, alpha=None):
    """alpha: optional per-class tensor, shape (num_classes,), broadcastable against
    targets (batch, num_classes). alpha_c close to 1 means 'upweight positives of this
    (rare) class'. Plain focal loss (alpha=None) only reweights easy-vs-hard examples;
    it does NOT compensate for a class simply being rare, which is the dominant
    imbalance in the 24-category factor taxonomy (positive rates from ~0.5% to ~45%).
    Empirically (see chat) plain focal loss recalls ~23% of a 3%-prevalence synthetic
    class; alpha=1-prevalence recalls 100% -- this is what class_weight='balanced' gave
    the classical TF-IDF baseline, ported to the focal-loss setting."""
    probs = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = probs * targets + (1 - probs) * (1 - targets)
    loss = ((1 - p_t) ** gamma) * ce
    if alpha is not None:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss
    return loss.mean()


def mean_pool(last_hidden_state, attention_mask):
    """Attention-mask-weighted mean over real (non-pad) tokens. RoBERTa's <s> token
    has no NSP-style pretraining objective pushing it to summarize the sequence the
    way BERT's [CLS] does, so for downstream classification a mean over all real
    tokens typically carries more usable signal than <s> alone (well established in
    the sentence-embedding literature, e.g. Sentence-BERT). Used for risk/factors
    pooling only -- evidence extraction already reads each token's own hidden state,
    so it has no pooling step to change."""
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class MultiTaskSuicideModel(nn.Module):
    def __init__(self, encoder, num_factors=NUM_FACTORS, freeze_bottom_fraction=0.5, dropout=0.2,
                 freeze_embeddings=True, use_mean_pooling=True):
        super().__init__()
        self.encoder = encoder
        self.use_mean_pooling = use_mean_pooling
        hidden = self.encoder.config.hidden_size

        if freeze_embeddings:
            for p in self.encoder.embeddings.parameters():
                p.requires_grad = False
        # freeze_embeddings should be False whenever freeze_bottom_fraction==0 (full
        # fine-tuning -- freezing just the embeddings while unfreezing everything else
        # is an inconsistent middle state) OR whenever FGM is enabled (it perturbs the
        # embedding gradient; frozen embeddings have no gradient to attack).
        #
        # freeze_bottom_fraction (not an absolute layer count) so this is correct
        # regardless of which backbone actually loads -- e.g. if a gated large model
        # falls back to a smaller one, "freeze the bottom half" should still mean half,
        # not silently freeze more (or all) of a shorter stack. n_layers is read from
        # the ACTUAL loaded encoder here, not assumed.
        n_layers = len(self.encoder.encoder.layer)
        n_freeze = int(round(freeze_bottom_fraction * n_layers))
        if n_freeze > 0:
            for layer in self.encoder.encoder.layer[: min(n_freeze, n_layers)]:
                for p in layer.parameters():
                    p.requires_grad = False

        self.dropout = nn.Dropout(dropout)

        # Pooled features (CLS concatenated with mean-pooled tokens, when
        # use_mean_pooling) go through a small MLP before the final heads, rather than
        # a bare Linear straight off the encoder -- a bit more capacity to combine the
        # two pooling views specifically for THESE tasks. Projects back down to `hidden`
        # so CoralHead and the optimizer's param-grouping (by name prefix "encoder")
        # need no changes -- this whole block is a "head", not the backbone.
        #
        # risk and factors get SEPARATE poolers (they shared one single pooler in the
        # previous round). The real submission's error analysis showed several factor
        # categories with a well-calibrated PREDICTED RATE that still had low F1 --
        # i.e. not a threshold problem, a genuine discrimination problem -- consistent
        # with the 24-way multi-label head and the 4-way ordinal head fighting over the
        # same projection's gradients. Splitting them costs very little (~1M extra
        # params on a ~125M-param model) and gives factors -- the still-clearly-lagging
        # task (0.42 vs top-team ~0.70) -- its own dedicated capacity and gradient path.
        pooled_in = hidden * 2 if use_mean_pooling else hidden

        def make_pooler():
            return nn.Sequential(
                nn.Linear(pooled_in, hidden),
                nn.GELU(),
                nn.LayerNorm(hidden),
                nn.Dropout(dropout),
            )

        self.risk_pooler = make_pooler()
        self.factors_pooler = make_pooler()

        self.risk_cls_head = nn.Linear(hidden, NUM_RISK_CLASSES)
        self.risk_coral_head = CoralHead(hidden, NUM_RISK_CLASSES)
        self.factors_head = nn.Linear(hidden, num_factors)
        self.evidence_head = nn.Linear(hidden, 2)

    def forward(self, input_ids, attention_mask):
        seq = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        cls = seq[:, 0]
        if self.use_mean_pooling:
            pooled_raw = torch.cat([cls, mean_pool(seq, attention_mask)], dim=-1)
        else:
            pooled_raw = cls
        pooled_raw = self.dropout(pooled_raw)
        risk_pooled = self.risk_pooler(pooled_raw)
        factors_pooled = self.factors_pooler(pooled_raw)
        return {
            "risk_cls": self.risk_cls_head(risk_pooled),
            "risk_coral": self.risk_coral_head(risk_pooled),
            "factors": self.factors_head(factors_pooled),
            "evidence": self.evidence_head(self.dropout(seq)),
        }

    def predict_risk(self, outputs, w=0.5):
        """Ensembles the two risk heads: w*p_cls + (1-w)*p_coral. w=0.5 (default)
        reproduces the fixed 50/50 blend every previous round used -- never actually
        tuned until now (see the CV cell for the pooled-OOF search over w)."""
        p_cls = F.softmax(outputs["risk_cls"], dim=-1)
        coral_probs = torch.sigmoid(outputs["risk_coral"])  # (batch, 3) = P(y>0),P(y>1),P(y>2)
        # convert CORAL cumulative probs to a proper per-class distribution
        p_gt = torch.cat([torch.ones_like(coral_probs[:, :1]), coral_probs], dim=1)   # P(y>=0..3 "and above")
        p_lt = torch.cat([coral_probs, torch.zeros_like(coral_probs[:, :1])], dim=1)  # shifted
        p_coral = p_gt - p_lt
        p_coral = p_coral.clamp(min=1e-6)
        p_coral = p_coral / p_coral.sum(dim=1, keepdim=True)
        p_final = w * p_cls + (1 - w) * p_coral
        return p_final.argmax(dim=-1), p_final


def compute_loss(model, outputs, risk_label, factors_label, evidence_label,
                  w_risk=0.4, w_evidence=0.3, w_factors=0.3, evidence_pos_weight=3.0,
                  factor_alpha=None):
    # REVERTED -- factors 0.4/evidence 0.2 was tried for one round: evidence_f1 dropped
    # (0.733, down from ~0.74-0.75) exactly as expected from taking its loss budget away,
    # but the real Subtask2 gain that was supposed to justify it did not show up (0.4554,
    # not clearly above the noise band around 0.4627-0.4530 established this session). A
    # confirmed cost without a confirmed benefit -- reverted to the original balance.
    risk_ce = F.cross_entropy(outputs["risk_cls"], risk_label, label_smoothing=0.1)
    risk_coral = model.risk_coral_head.loss(outputs["risk_coral"], risk_label)
    risk_loss = 0.5 * risk_ce + 0.5 * risk_coral

    factors_loss = focal_bce(outputs["factors"], factors_label, gamma=2.0, alpha=factor_alpha)

    ev_logits = outputs["evidence"].reshape(-1, 2)
    ev_labels = evidence_label.reshape(-1)
    class_w = torch.tensor([1.0, evidence_pos_weight], device=ev_logits.device)
    evidence_loss = F.cross_entropy(ev_logits, ev_labels, ignore_index=-100, weight=class_w)

    total = w_risk * risk_loss + w_evidence * evidence_loss + w_factors * factors_loss
    parts = {"risk": risk_loss.item(), "evidence": evidence_loss.item(),
             "factors": factors_loss.item(), "total": total.item()}
    return total, parts


class EMA:
    """Exponential moving average of trainable weights. apply_shadow()/restore() are
    used only around evaluation -- training always continues from the REAL (non-EMA)
    weights, never from the shadow, which is the standard and correct usage pattern."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.data, alpha=1 - self.decay)

    def apply_shadow(self, model):
        self.backup = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.shadow[n])

    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n])
        self.backup = {}


class FGM:
    """Fast Gradient Method adversarial training (Miyato et al.), perturbing the word
    embedding layer along its own gradient direction. Requires embeddings to be
    trainable (see the freezing note in MultiTaskSuicideModel.__init__) -- attack()
    silently does nothing if no matching parameter has a gradient, rather than
    crashing, but that means it's providing zero benefit in that case."""

    def __init__(self, model, emb_name="word_embeddings", epsilon=1.0):
        self.model = model
        self.emb_name = emb_name
        self.epsilon = epsilon
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and torch.isfinite(norm):
                    param.data.add_(self.epsilon * param.grad / norm)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}


def extract_evidence_spans(pred_tags, offset_mapping, text, merge_gap=1):
    """pred_tags: 1D array of 0/1 per token (already masked to real tokens only).
    offset_mapping: list of (char_start, char_end) per token, aligned to pred_tags.
    Merges runs separated by a gap of <= merge_gap non-evidence tokens (reduces the
    fragmentation that hurt the classical token-classifier baseline's one-to-one
    Phrase-F1 matching), then reconstructs verbatim substrings via char offsets."""
    n = len(pred_tags)
    tags = list(pred_tags)
    i = 0
    while i < n:
        if tags[i] == 0:
            j = i
            while j < n and tags[j] == 0:
                j += 1
            gap = j - i
            if 0 < gap <= merge_gap and i > 0 and j < n and tags[i - 1] == 1 and tags[j] == 1:
                for k in range(i, j):
                    tags[k] = 1
            i = j
        else:
            i += 1
    spans = []
    i = 0
    while i < n:
        if tags[i] == 1:
            j = i
            while j < n and tags[j] == 1:
                j += 1
            start = offset_mapping[i][0]
            end = offset_mapping[j - 1][1]
            if end > start:
                spans.append(text[start:end])
            i = j
        else:
            i += 1
    return spans

## Data preparation
Aligns gold evidence spans to token-level labels using the tokenizer's own character offset mapping (`return_offsets_mapping=True`), so this works correctly regardless of how RoBERTa's BPE splits words — no hand-rolled word tokenizer needed here, unlike the classical baseline.

In [ ]:
import numpy as np

RISK_TO_IDX = {r: i for i, r in enumerate(RISK_LEVELS)}


def find_span_char_range(post, span, start_from=0):
    post_low = post.lower()
    span_low = span.strip().lower()
    if not span_low:
        return None
    idx = post_low.find(span_low, start_from)
    if idx == -1:
        return None
    return (idx, idx + len(span_low))


def gold_char_ranges(post, evidence_spans):
    ranges = []
    for span in evidence_spans:
        # search past the end of the previous match of the *same* span text so repeated
        # phrases in one post don't all collapse onto the first occurrence
        prior_ends = [e for (s, e) in ranges]
        start_from = max(prior_ends) if prior_ends else 0
        r = find_span_char_range(post, span, start_from) or find_span_char_range(post, span, 0)
        if r:
            ranges.append(r)
    return ranges


def align_evidence_labels(offset_mapping, gold_ranges):
    """offset_mapping: list of (char_start, char_end) per token (special tokens are (0,0)).
    Returns a list of labels, one per token: 1=evidence, 0=not, -100=special/pad token."""
    labels = []
    for (s, e) in offset_mapping:
        if s == 0 and e == 0:
            labels.append(-100)
            continue
        is_ev = any(max(s, gs) < min(e, ge) for (gs, ge) in gold_ranges)
        labels.append(1 if is_ev else 0)
    return labels


def factors_multihot(factors_list):
    vec = np.zeros(len(FACTOR_TAXONOMY), dtype=np.float32)
    idx = {f: i for i, f in enumerate(FACTOR_TAXONOMY)}
    for f in factors_list:
        if f in idx:
            vec[idx[f]] = 1.0
    return vec


def prepare_example(post, risk, factors, evidence_spans, tokenizer, max_length=512):
    enc = tokenizer(post, truncation=True, max_length=max_length,
                     padding="max_length", return_offsets_mapping=True)
    gold_ranges = gold_char_ranges(post, evidence_spans)
    ev_labels = align_evidence_labels(enc["offset_mapping"], gold_ranges)
    # also ignore pure padding positions (attention_mask == 0) beyond what offset (0,0) already covers
    for i, m in enumerate(enc["attention_mask"]):
        if m == 0:
            ev_labels[i] = -100
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "offset_mapping": enc["offset_mapping"],
        "risk_label": RISK_TO_IDX[risk],
        "factors_label": factors_multihot(factors),
        "evidence_label": ev_labels,
    }

## Training utilities

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


def fit_classical_factors_model(train_texts, train_factor_lists, taxonomy_size):
    """TF-IDF + one logistic regression per category, class_weight='balanced' -- the same
    approach the very first classical baseline in this project used, before any of the
    transformer work. Fit fresh per fold on that fold's own (oversampled) training posts,
    mirroring the neural model's own fold split exactly -- no extra leakage risk. Fast
    (seconds): this exists to blend with the neural model afterward (see the CV cell), on
    the theory that a lexical/n-gram model can carry different signal than a fine-tuned
    transformer, especially for categories with too few examples for the transformer to
    generalize well but where a keyword association still shows up reliably."""
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
    X = vectorizer.fit_transform(train_texts)
    Y = np.stack([factors_multihot(labels) for labels in train_factor_lists])
    models = []
    for c in range(taxonomy_size):
        y_c = Y[:, c]
        if y_c.sum() < 2:  # LogisticRegression needs both classes present
            models.append(None)
            continue
        clf = LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0)
        clf.fit(X, y_c)
        models.append(clf)
    return vectorizer, models


def predict_classical_factors(vectorizer, models, texts, taxonomy_size):
    X = vectorizer.transform(texts)
    probs = np.zeros((len(texts), taxonomy_size))
    for c, clf in enumerate(models):
        if clf is None:
            continue
        probs[:, c] = clf.predict_proba(X)[:, 1]
    return probs


def get_oversampled_indices(indices, factors_list, rare_threshold_count=30, oversample_factor=3):
    """indices: array of row positions (e.g. a fold's training-portion indices).
    factors_list: the FULL train_df factors list, indexed by original row position.
    Rows containing at least one category with fewer than rare_threshold_count positive
    examples WITHIN this index set get repeated oversample_factor times total; everything
    else appears once. Only ever apply this to a TRAINING split -- oversampling a
    validation split would double-count rows in metrics."""
    cat_counts = {}
    for i in indices:
        for f in set(factors_list[i]):
            cat_counts[f] = cat_counts.get(f, 0) + 1
    rare_cats = {c for c, n in cat_counts.items() if n < rare_threshold_count}

    expanded = []
    for i in indices:
        expanded.append(i)
        if rare_cats & set(factors_list[i]):
            expanded.extend([i] * (oversample_factor - 1))
    return np.array(expanded), rare_cats


class SuicideRiskDataset(Dataset):
    def __init__(self, posts, risks, factors, evidences, tokenizer, max_length=512):
        self.examples = [
            prepare_example(p, r, f, e, tokenizer, max_length)
            for p, r, f, e in zip(posts, risks, factors, evidences)
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        return {
            "input_ids": torch.tensor(ex["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(ex["attention_mask"], dtype=torch.long),
            "risk_label": torch.tensor(ex["risk_label"], dtype=torch.long),
            "factors_label": torch.tensor(ex["factors_label"], dtype=torch.float),
            "evidence_label": torch.tensor(ex["evidence_label"], dtype=torch.long),
        }


def make_optimizer(model, backbone_lr=2e-5, head_lr=1e-4, weight_decay=0.01):
    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("encoder")]
    head_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder")]
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": backbone_lr},
        {"params": head_params, "lr": head_lr},
    ], weight_decay=weight_decay)


def linear_warmup_schedule(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(step):
        if step < num_warmup_steps:
            return step / max(1, num_warmup_steps)
        return max(0.0, (num_training_steps - step) / max(1, num_training_steps - num_warmup_steps))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def compute_factor_alpha(factors_label_matrix, device, clip=(0.15, 0.85)):
    """alpha_c = 1 - prevalence_c, clipped. A synthetic 24-category test (matching this
    dataset's real prevalence spread, see chat) showed the original (0.05, 0.95) clip
    overcorrects -- it fixes rare-category recall but pushes MID-frequency categories
    into broad over-prediction (macro F1 0.690). (0.15, 0.85) recovers most of the
    rare-category benefit while avoiding the overcorrection (macro F1 0.843 in the
    same test) -- this is now the default."""
    prevalence = np.asarray(factors_label_matrix).mean(axis=0)
    alpha = 1.0 - prevalence
    alpha = np.clip(alpha, clip[0], clip[1])
    return torch.tensor(alpha, dtype=torch.float32, device=device)


def _best_threshold(probs_c, y_true, cands, beta2):
    best_score, best_t = -1.0, 0.5
    for t in cands:
        pred = (probs_c > t).astype(float)
        tp = ((pred == 1) & (y_true == 1)).sum()
        fp = ((pred == 1) & (y_true == 0)).sum()
        fn = ((pred == 0) & (y_true == 1)).sum()
        prec = tp / (tp + fp) if tp + fp > 0 else 0.0
        rec = tp / (tp + fn) if tp + fn > 0 else 0.0
        denom = beta2 * prec + rec
        score = (1 + beta2) * prec * rec / denom if denom > 0 else 0.0
        if score > best_score:
            best_score, best_t = score, t
    return best_t


def tune_factor_thresholds(factor_probs, factor_labels, taxonomy_size, min_support=15, fbeta=1.0,
                            overpredict_ratio=2.0, category_names=None, verbose=True):
    """Per-category decision threshold that maximizes F-beta against the given
    probabilities and labels. Called ONCE on pooled out-of-fold predictions across the
    FULL train set (see the CV cell) rather than per-fold on a small dev split --
    pooling gives every category far more positive examples to search a threshold
    against, which matters a lot for the rarer categories. A category needs at least
    `min_support` positive examples in the POOLED set before we trust a searched
    threshold for it; below that, 0.5 stays.

    fbeta<1 weights precision more than recall, applied PER CATEGORY via a two-pass
    check: first find the plain-F1 (beta=1) optimal threshold for a category, then
    look at the predicted rate AT THAT THRESHOLD -- only if it still exceeds
    overpredict_ratio x true prevalence does the category get re-searched with the
    precision-biased fbeta. An earlier version checked over-prediction at the raw,
    UNTUNED 0.5 threshold instead, which is the wrong reference point: rare
    categories' sigmoid outputs cluster low just from the class-imbalance base rate,
    so their F1-optimal threshold typically sits well below 0.5 -- a category can
    look perfectly calibrated at a fixed 0.5 while still over-predicting 4-5x at the
    threshold actually being used. Checking at 0.5 caught only 1/24 categories in
    practice and missed clear over-predictors like "meaning in life" (4.7x its true
    rate at its own F1-optimal point). Checking at each category's own natural
    operating point is the correct reference for "is THIS category over-predicting."
    factor_probs/factor_labels: numpy arrays, shape (N, taxonomy_size)."""
    factor_probs = np.asarray(factor_probs)
    factor_labels = np.asarray(factor_labels)
    thresholds = np.full(taxonomy_size, 0.5)
    overpredicting = []
    for c in range(taxonomy_size):
        y_true = factor_labels[:, c]
        support = y_true.sum()
        if support < min_support:
            continue
        probs_c = factor_probs[:, c]
        true_rate = y_true.mean()
        cands = np.unique(probs_c)

        f1_opt_t = _best_threshold(probs_c, y_true, cands, beta2=1.0)
        f1_opt_rate = (probs_c > f1_opt_t).mean()
        is_overpredicting = true_rate > 0 and f1_opt_rate > overpredict_ratio * true_rate

        if is_overpredicting:
            thresholds[c] = _best_threshold(probs_c, y_true, cands, beta2=fbeta ** 2)
            overpredicting.append(c)
        else:
            thresholds[c] = f1_opt_t
    if verbose and fbeta != 1.0:
        names = [category_names[c] for c in overpredicting] if category_names else overpredicting
        print(f"  factor threshold tuning: F-beta={fbeta} applied to {len(names)}/{taxonomy_size} "
              f"categories over-predicting (>{overpredict_ratio}x true rate) at their own F1-optimal point: {names}")
    return thresholds


def tune_evidence_hparams(evidence_probs_list, offsets_list, posts, true_evidence_list,
                           thresholds=(0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95), merge_gaps=(0, 1, 2, 3)):
    """Grid-search (threshold, merge_gap) directly against the competition's own
    Phrase-F1 metric, rather than using the fixed defaults (0.5, 1) the token classifier
    was trained with. Token-level BCE and phrase-level Phrase-F1 are related but not
    identical -- over-merging (merge_gap too high) can weld separate evidence phrases
    into one span that exceeds the 3x-length cap and matches nothing; under-merging
    fragments a single phrase into pieces that individually fail the containment check.
    Called once on pooled out-of-fold predictions (see the CV cell), same spirit as the
    factor-threshold tuning above -- a few dozen cheap combinations, no retraining."""
    best_f1, best_t, best_g = -1.0, 0.5, 1
    for t in thresholds:
        for g in merge_gaps:
            preds = []
            for ev_p, offs, post in zip(evidence_probs_list, offsets_list, posts):
                tags = (np.array(ev_p) > t).astype(int).tolist()
                preds.append(extract_evidence_spans(tags, offs, post, merge_gap=g))
            f1 = phrase_f1_corpus(preds, true_evidence_list)
            if f1 > best_f1:
                best_f1, best_t, best_g = f1, t, g
    return best_f1, best_t, best_g


def train_one_epoch(model, loader, optimizer, scheduler, device, use_amp=False, scaler=None,
                     factor_alpha=None, ema=None, fgm=None):
    model.train()
    total = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        if use_amp and scaler is not None:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                out = model(batch["input_ids"], batch["attention_mask"])
                loss, _ = compute_loss(model, out, batch["risk_label"], batch["factors_label"],
                                        batch["evidence_label"], factor_alpha=factor_alpha)
            scaler.scale(loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    out_adv = model(batch["input_ids"], batch["attention_mask"])
                    loss_adv, _ = compute_loss(model, out_adv, batch["risk_label"], batch["factors_label"],
                                                batch["evidence_label"], factor_alpha=factor_alpha)
                scaler.scale(loss_adv).backward()
                fgm.restore()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            # GradScaler skips the actual optimizer step on iterations where it detects
            # gradient overflow (common in the first few steps while it calibrates) --
            # when that happens, scale_after < scale_before, and the scheduler must skip
            # in lockstep too, or the LR schedule silently drifts out of sync with the
            # real number of optimization steps taken.
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            step_was_skipped = scaler.get_scale() < scale_before
        else:
            out = model(batch["input_ids"], batch["attention_mask"])
            loss, _ = compute_loss(model, out, batch["risk_label"], batch["factors_label"],
                                    batch["evidence_label"], factor_alpha=factor_alpha)
            loss.backward()
            if fgm is not None:
                fgm.attack()
                out_adv = model(batch["input_ids"], batch["attention_mask"])
                loss_adv, _ = compute_loss(model, out_adv, batch["risk_label"], batch["factors_label"],
                                            batch["evidence_label"], factor_alpha=factor_alpha)
                loss_adv.backward()
                fgm.restore()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            step_was_skipped = False
        if ema is not None:
            ema.update(model)
        if not step_was_skipped:
            scheduler.step()
        total += loss.item()
    return total / max(1, len(loader))


@torch.no_grad()
def predict_probs(model, posts, tokenizer, device, max_length=512, risk_weight=0.5,
                   return_risk_components=False):
    """Returns per-post risk class-probabilities (N,4) blended at risk_weight, factor
    probabilities (N,24), and per-token evidence-probabilities together with
    offsets/attn (for later ensembling and span extraction) -- rather than hard
    labels, so multiple folds can be averaged.

    return_risk_components=True additionally returns the raw, UNBLENDED risk_cls and
    risk_coral probabilities (each N,4) as two extra return values -- lets the CV cell
    pool them out-of-fold and search for the best blend weight after the fact, without
    needing a second encoder forward pass per candidate weight (w=1.0/w=0.0 through
    predict_risk isolate each component from the SAME already-computed outputs, so
    this only adds cheap tensor arithmetic, not extra model calls)."""
    model.eval()
    risk_probs, factor_probs, evidence_probs, offsets_list = [], [], [], []
    risk_cls_probs, risk_coral_probs = [], []
    for post in posts:
        enc = tokenizer(post, truncation=True, max_length=max_length, padding="max_length",
                         return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping")[0].tolist()
        input_ids = enc["input_ids"].to(device)
        attn = enc["attention_mask"].to(device)
        out = model(input_ids, attn)
        _, p_risk = model.predict_risk(out, w=risk_weight)
        risk_probs.append(p_risk[0].cpu().numpy())
        if return_risk_components:
            _, p_cls_full = model.predict_risk(out, w=1.0)
            _, p_coral_full = model.predict_risk(out, w=0.0)
            risk_cls_probs.append(p_cls_full[0].cpu().numpy())
            risk_coral_probs.append(p_coral_full[0].cpu().numpy())
        factor_probs.append(torch.sigmoid(out["factors"])[0].cpu().numpy())
        real_len = int(attn[0].sum().item())
        ev_p = torch.softmax(out["evidence"][0, :real_len], dim=-1)[:, 1].cpu().numpy()  # P(evidence)
        evidence_probs.append(ev_p)
        offsets_list.append(offsets[:real_len])
    result = (np.stack(risk_probs), np.stack(factor_probs), evidence_probs, offsets_list)
    if return_risk_components:
        result = result + (np.stack(risk_cls_probs), np.stack(risk_coral_probs))
    return result


def decode_predictions(risk_probs, factor_probs, evidence_probs, offsets_list, posts,
                        risk_threshold_none=False, factor_threshold=0.5, evidence_threshold=0.5,
                        evidence_merge_gap=1):
    risk_preds = [RISK_LEVELS[i] for i in risk_probs.argmax(axis=1)]
    ft = np.asarray(factor_threshold)
    if ft.ndim == 0:
        ft = np.full(len(FACTOR_TAXONOMY), float(ft))
    factor_preds = [
        [FACTOR_TAXONOMY[i] for i in range(len(FACTOR_TAXONOMY)) if row[i] > ft[i]]
        for row in factor_probs
    ]
    evidence_preds = []
    for ev_p, offsets, post in zip(evidence_probs, offsets_list, posts):
        tags = (np.array(ev_p) > evidence_threshold).astype(int).tolist()
        evidence_preds.append(extract_evidence_spans(tags, offsets, post, merge_gap=evidence_merge_gap))
    return risk_preds, factor_preds, evidence_preds


def evaluate_fold(risk_probs, factor_probs, evidence_probs, offsets_list, posts,
                   true_risk, true_factors, true_evidence):
    risk_pred, factor_pred, evidence_pred = decode_predictions(
        risk_probs, factor_probs, evidence_probs, offsets_list, posts)
    rf1 = weighted_f1_risk(true_risk, risk_pred)
    ff1 = macro_f1_factors(true_factors, factor_pred)
    ef1 = phrase_f1_corpus(evidence_pred, true_evidence)
    comp = composite_score(rf1, ef1, ff1)
    return dict(risk_f1=rf1, evidence_f1=ef1, factors_f1=ff1, composite=comp)

## LLM-based factors signal (zero-shot, local, no API key)

A small (3.8B) instruction-tuned model (`microsoft/Phi-3.5-mini-instruct`, MIT-licensed, no gate) reads each post once and lists which of the 24 factor categories it thinks apply, using only its own pretrained world/clinical knowledge — no fine-tuning, no exposure to training labels. This is a genuinely different lever from everything else in this notebook: the neural and classical models can only learn a category from examples of it in the 1635-row training set, and several categories have single-digit-to-low-double-digit example counts total. An LLM does not need dozens of examples to recognize a well-known concept like "substance use" or "interpersonal violence" from context — the theory is that this helps most exactly where the other two models are weakest (the rare categories that have been stuck near 0 F1 all session).

Runs ONCE over every post (train + leaderboard combined, ~2000 posts) before the fold loop starts, not per-fold. Expect roughly 30–60 minutes depending on the Colab GPU's exact throughput — this is the slowest single cell in the notebook. If it fails for any reason (OOM, load error, unexpected output format), it prints a warning and continues with the LLM signal simply unused — the blending logic in the CV cell below automatically falls back to the neural+classical blend from last round, so a failure here does not break anything downstream. If you hit an out-of-memory error, lowering `batch_size` in the call below is the first thing to try.

In [ ]:
import re
import torch
from transformers import pipeline

LLM_MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"  # MIT-licensed, no gate, ~3.8B params, fits
                                                       # comfortably in fp16 on a T4 (~8GB)

FACTOR_DESCRIPTIONS = {
    "mental health issues": "a diagnosed or described mental health condition or symptoms, e.g. depression, anxiety, PTSD",
    "physical health/characteristic": "a physical illness, disability, chronic pain, or physical appearance concern",
    "substance use": "alcohol or drug use, misuse, or dependence",
    "hopelessness": "feeling hopeless, worthless, or that things will not improve",
    "emotion dysregulation": "intense, unstable, or hard-to-control emotions",
    "low self-esteem": "a negative self-image or lack of self-confidence",
    "poor school performance": "academic struggles, failing grades, or school-related stress",
    "low socio-economic status": "poverty, financial hardship, or unemployment",
    "interpersonal violence": "abuse, assault, or violence from another person (physical, sexual, or emotional)",
    "prior self-harm or suicidal thought/attempt": "a PAST history of self-harm, suicidal thoughts, or a suicide attempt (not the current post's own risk level)",
    "poor social support": "lack of friends, isolation, or feeling unsupported by others",
    "interpersonal difficulty": "conflict or problems in a relationship (friend, romantic partner, family)",
    "dysfunctional family": "family conflict, instability, or neglect",
    "exposure to others' suicide": "knowing someone (personally) who died by or attempted suicide",
    "stressful life event": "a specific recent stressor, e.g. a breakup, job loss, or move",
    "traumatic experience": "a past traumatic event, e.g. abuse, accident, or disaster",
    "cognitive deficits": "difficulty thinking clearly, concentrating, or a cognitive impairment",
    "suicide means (with access)": "mentions having access to a specific method for suicide",
    "sexual orientation related issues": "distress or conflict related to sexual orientation or gender identity",
    "social support": "PROTECTIVE factor -- mentions of supportive friends, family, or community currently present",
    "coping strategy": "PROTECTIVE factor -- mentions of a specific way the person copes with or manages distress",
    "psychological capital": "PROTECTIVE factor -- resilience, optimism, or self-efficacy",
    "sense of responsibility": "PROTECTIVE factor -- feeling responsible for someone else (e.g. children, family) as a reason to keep going",
    "meaning in life": "PROTECTIVE factor -- a sense of purpose or meaning",
}

SYSTEM_PROMPT = (
    "You are a careful annotator for a mental-health research dataset. Given a social media "
    "post, decide which of the following factors are clearly present in the post, based only "
    "on its text. Categories:\n"
    + "\n".join(f"- {name}: {desc}" for name, desc in FACTOR_DESCRIPTIONS.items())
    + "\n\nRespond with ONLY a comma-separated list of the category names EXACTLY as written "
    "above that clearly apply, or the single word 'none' if none clearly apply. Do not explain "
    "your reasoning. Do not include categories that are only weakly implied."
)


def parse_llm_factors(response_text, taxonomy):
    """Fuzzy-matches the model's free-text response back to the fixed taxonomy -- substring
    containment rather than exact match, to tolerate minor wording drift (e.g. 'substance
    abuse' vs 'substance use') without crashing on anything unparseable."""
    if not response_text or response_text.strip().lower().startswith("none"):
        return []
    found = []
    resp_lower = response_text.lower()
    for cat in taxonomy:
        if cat.lower() in resp_lower:
            found.append(cat)
    return found


def run_llm_factor_classifier(posts, taxonomy, model_name=LLM_MODEL_NAME, batch_size=8, max_new_tokens=150):
    """Returns a (len(posts), len(taxonomy)) 0/1 array. One pass, zero-shot, no fitting step --
    safe to call once on the full train+leaderboard set with no leakage concern, since it never
    sees any label, only post text."""
    pipe = pipeline(
        "text-generation", model=model_name, torch_dtype=torch.float16,
        device_map="cuda" if torch.cuda.is_available() else "cpu", trust_remote_code=True,
    )
    probs = np.zeros((len(posts), len(taxonomy)))
    gen_args = dict(max_new_tokens=max_new_tokens, do_sample=False,
                     return_full_text=False, pad_token_id=pipe.tokenizer.eos_token_id)
    for start in range(0, len(posts), batch_size):
        batch_posts = posts[start:start + batch_size]
        messages_batch = [
            [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": p}]
            for p in batch_posts
        ]
        outputs = pipe(messages_batch, batch_size=batch_size, **gen_args)
        for j, out in enumerate(outputs):
            text = out[0]["generated_text"] if isinstance(out, list) else out["generated_text"]
            for cat in parse_llm_factors(text, taxonomy):
                probs[start + j, taxonomy.index(cat)] = 1.0
        if (start // batch_size) % 10 == 0:
            print(f"  LLM factor classification: {min(start + batch_size, len(posts))}/{len(posts)} posts")
    del pipe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return probs


llm_factor_probs_train, llm_factor_probs_lb = None, None
try:
    print("Loading LLM for zero-shot factor classification (this cell can take 30-60 min)...")
    all_posts_for_llm = train_df["post_clean"].tolist() + leaderboard_df["post_clean"].tolist()
    llm_factor_probs_all = run_llm_factor_classifier(all_posts_for_llm, FACTOR_TAXONOMY)
    llm_factor_probs_train = llm_factor_probs_all[:len(train_df)]
    llm_factor_probs_lb = llm_factor_probs_all[len(train_df):]
    print(f"done -- mean factors flagged per post: {llm_factor_probs_all.sum(axis=1).mean():.2f} "
          f"(training set's true mean is ~3.5)")
except Exception as e:
    print(f"[warning] LLM factor classification failed ({type(e).__name__}: {e}) -- continuing "
          f"without it. The CV cell below automatically falls back to the neural+classical blend.")

Grouped by `anon_user_id` (never GroupKFold-naively-by-row) — this mirrors
the real leaderboard split, which has zero user overlap with train, so these
CV numbers should be a reasonably honest estimate of leaderboard performance.

**Runtime note:** with `FREEZE_BOTTOM_FRACTION=0.5` (the setting these timings were measured at -- see config for the current value), each fold
has consistently taken ~30s/epoch, ~5 min/fold at `EPOCHS=10` — measured from
several actual runs, not an estimate. At `N_FOLDS=8` that's roughly 40-45 min
total, comfortably inside a free-tier Colab session. An earlier full-unfreeze
experiment was noticeably slower, 2-5 min/epoch; that's not the current
setting, so it no longer applies here.

**Truncation note:** posts over ~380 words get truncated at 512 tokens, so
any evidence spans past that point can't be recovered. Only a small tail of
posts are this long (median post length is ~38 words), so this mainly
affects a handful of very long posts.

**Threshold/hyperparameter tuning has moved:** earlier versions carved a 15%
dev split out of each fold's training portion purely to tune factor
thresholds, then averaged 5 per-fold thresholds. That's gone — each fold now
trains on its full portion, and calibration (factor thresholds AND the
evidence threshold/merge_gap) happens ONCE after the loop, on out-of-fold
predictions pooled across all folds. See the cell right after the loop for
why that's more reliable, especially for rare factor categories.

**This round:** `N_FOLDS` 5→8 — not a code/logic change, an ensemble-size
increase. Three straight rounds of byte-identical factor-threshold code
produced real Subtask2 scores ranging 0.4252-0.4530 despite a stable
pooled-OOF estimate (0.445-0.446 throughout) — real run-to-run variance the
OOF metric wasn't catching, most likely training non-determinism plus the
378-row leaderboard set being a small sample for a 24-way macro average. More
ensemble members smooths the averaged leaderboard probabilities; it won't
eliminate the variance, but should reduce it.

**Previous round:** `EPOCHS` 8→10, and the factor threshold search started
optimizing F-beta rather than plain F1, applied uniformly to every category.
That real submission came back WORSE on factors (0.4184→0.4085) despite a
better pooled-OOF estimate (0.3968→0.4301): it fixed genuinely over-predicting
categories but also pushed already-calibrated, high-prevalence ones (e.g.
`hopelessness`) toward needless under-prediction, and macro F1 weighs every
category equally, so that was enough to flip the net effect negative.

**Previous round:** F-beta applied per category, gated on over-prediction at
a fixed 0.5 threshold. That real submission WAS a net improvement (0.4184→
0.4252 on Subtask2) but the gate only caught 1/24 categories — checking at a
fixed 0.5 is the wrong reference point for rare categories, whose F1-optimal
threshold typically sits well below 0.5 from the class-imbalance base rate
alone, so a category can look fine at 0.5 while still over-predicting 4-5x at
the threshold actually in use ("meaning in life" was missed this way).

**This round:** Subtask1 (risk+evidence) had been flat across four real
submissions (0.7482/0.7482/0.7475/0.7478) despite CV risk_f1 visibly climbing
each round -- so this round leaves factors alone and tunes the one risk-side
knob that was never touched: the fixed 50/50 blend between the CLS head and
the CORAL head. It's now searched the same way as the factor thresholds and
evidence hyperparameters -- pooled OOF, printed before/after so you can see
whether it actually helped before trusting it. The evidence threshold grid is
also widened (up to 0.9, was capped at 0.7) since the tuned value has landed
at the top of the old range for three rounds running.

**Bug fixed this round:** the large-model config change (`FREEZE_BOTTOM_LAYERS=12`,
sized for a 24-layer model) silently broke the fallback path -- when the gate
wasn't accepted and the run fell back to the 12-layer base model, 12 froze
ALL of it, not half, leaving only the heads trainable on a fully static
encoder. Loss plateaued at 0.34 instead of the usual ~0.14, and Attempt-class
recall collapsed to 0.20. Fixed properly, not just patched: freezing is now
stored as a FRACTION (`FREEZE_BOTTOM_FRACTION`) and computed against whatever
backbone actually loads, so this class of bug can't recur on a future
backbone swap or fallback. The CV cell now also prints how many layers it
froze on fold 0, specifically so this is visible immediately rather than only
showing up as an unexplained score crash three cells later.

In [ ]:
splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
X_text = train_df["post_clean"].values
y_risk = train_df["risk_clean"].values
y_factors = train_df["factors_clean"].tolist()
y_evidence = train_df["evidence_clean"].tolist()
groups = train_df["anon_user_id"].values

# resolve the backbone ONCE, with a graceful fallback if the gated model hasn't been
# unlocked yet -- tokenizer files are small, so this is a cheap canary for whether the
# larger model download (below, once per fold) will also succeed.
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    ACTIVE_MODEL_NAME = MODEL_NAME
except Exception as e:
    print(f"[warning] could not load '{MODEL_NAME}' ({type(e).__name__}: {e})")
    print(f"          falling back to '{MODEL_NAME_FALLBACK}'. To use the domain-adapted")
    print(f"          backbone: accept the license at https://huggingface.co/{MODEL_NAME} ,")
    print("          then re-run the login() cell above with a token that has access, then re-run this cell.")
    ACTIVE_MODEL_NAME = MODEL_NAME_FALLBACK
    tokenizer = AutoTokenizer.from_pretrained(ACTIVE_MODEL_NAME)
print("backbone in use:", ACTIVE_MODEL_NAME)

fold_results = []
lb_risk_cls_probs_sum = np.zeros((len(leaderboard_df), 4))
lb_risk_coral_probs_sum = np.zeros((len(leaderboard_df), 4))
lb_factor_probs_sum = np.zeros((len(leaderboard_df), len(FACTOR_TAXONOMY)))
lb_evidence_probs_sum = None
lb_offsets_ref = None

# out-of-fold predictions/probabilities across ALL training rows. Used both for the
# error-analysis cell below AND for the pooled global calibration that replaces the
# old per-fold dev-split tuning -- see the cell after the loop. risk_cls/risk_coral
# are kept SEPARATE (not pre-blended) so the ensemble weight between the two risk
# heads can be searched after the loop too, same pattern as the factor thresholds
# and evidence hyperparameters -- it was the one risk/evidence knob never tuned.
oof_risk_true, oof_risk_pred = [None]*len(train_df), [None]*len(train_df)
oof_risk_cls_probs = np.zeros((len(train_df), 4))
oof_risk_coral_probs = np.zeros((len(train_df), 4))
oof_factors_true, oof_factors_pred = [None]*len(train_df), [None]*len(train_df)
oof_factor_probs = np.zeros((len(train_df), len(FACTOR_TAXONOMY)))
oof_classical_factor_probs = np.zeros((len(train_df), len(FACTOR_TAXONOMY)))
oof_evidence_probs = [None]*len(train_df)
oof_evidence_offsets = [None]*len(train_df)

lb_classical_factor_probs_sum = np.zeros((len(leaderboard_df), len(FACTOR_TAXONOMY)))

for fold, (tr_idx, va_idx) in enumerate(splitter.split(X_text, y_risk, groups)):
    print(f"\n{'='*60}\nFOLD {fold}\n{'='*60}")
    t0 = time.time()

    # oversample rows containing rare factor categories -- applied to the fold's full
    # training portion. (Earlier versions carved a dev split out of tr_idx here, used
    # only for threshold tuning; that's gone -- see the note after the loop for why
    # pooling out-of-fold validation predictions across all folds instead is both
    # simpler and more reliable. tr_idx now trains on 100% of its rows.)
    tr_idx_oversampled, rare_cats = get_oversampled_indices(
        tr_idx, y_factors, rare_threshold_count=RARE_FACTOR_THRESHOLD, oversample_factor=OVERSAMPLE_FACTOR)
    if fold == 0:
        print(f"  oversampling: {len(tr_idx)} -> {len(tr_idx_oversampled)} training rows "
              f"({len(tr_idx_oversampled)/len(tr_idx):.2f}x) for {len(rare_cats)} rare categories: {sorted(rare_cats)}")

    fit_ds = SuicideRiskDataset(X_text[tr_idx_oversampled], y_risk[tr_idx_oversampled],
                                 [y_factors[i] for i in tr_idx_oversampled], [y_evidence[i] for i in tr_idx_oversampled],
                                 tokenizer, MAX_LENGTH)
    loader = DataLoader(fit_ds, batch_size=BATCH_SIZE, shuffle=True)

    # per-class alpha (1 - prevalence, computed from the ORIGINAL fold labels,
    # deliberately BEFORE oversampling) -- counteracts class-PREVALENCE imbalance, which
    # plain focal loss does not address on its own. Computing this from the OVERSAMPLED
    # set instead would make rare categories look artificially less rare (since we just
    # duplicated their rows), which would partially cancel the oversampling's own effect --
    # two mechanisms fighting each other instead of working together. Using the true,
    # pre-oversampling prevalence keeps them complementary.
    fold_factor_matrix = np.stack([factors_multihot(y_factors[i]) for i in tr_idx])
    factor_alpha = compute_factor_alpha(fold_factor_matrix, device)
    if fold == 0:
        rarest = np.argsort(fold_factor_matrix.mean(axis=0))[:3]
        print("  factor alpha range: [{:.2f}, {:.2f}] -- rarest categories this fold:".format(
            factor_alpha.min().item(), factor_alpha.max().item()))
        for i in rarest:
            print(f"    {FACTOR_TAXONOMY[i]!r}: prevalence {fold_factor_matrix[:,i].mean():.3f}, alpha {factor_alpha[i].item():.2f}")

    encoder = AutoModel.from_pretrained(ACTIVE_MODEL_NAME)
    model = MultiTaskSuicideModel(encoder, num_factors=len(FACTOR_TAXONOMY),
                                   freeze_bottom_fraction=FREEZE_BOTTOM_FRACTION,
                                   freeze_embeddings=FREEZE_EMBEDDINGS,
                                   use_mean_pooling=USE_MEAN_POOLING).to(device)
    if fold == 0:
        n_layers_loaded = len(encoder.encoder.layer)
        print(f"  backbone has {n_layers_loaded} layers; freezing bottom "
              f"{int(round(FREEZE_BOTTOM_FRACTION * n_layers_loaded))} of them "
              f"({FREEZE_BOTTOM_FRACTION:.0%})")

    optimizer = make_optimizer(model, backbone_lr=BACKBONE_LR, head_lr=HEAD_LR)
    total_steps = len(loader) * EPOCHS
    scheduler = linear_warmup_schedule(optimizer, num_warmup_steps=int(0.1*total_steps),
                                        num_training_steps=total_steps)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
    ema = EMA(model, decay=EMA_DECAY) if USE_EMA else None
    fgm = FGM(model, epsilon=FGM_EPSILON) if USE_FGM else None

    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, loader, optimizer, scheduler, device,
                                use_amp=(device.type == "cuda"), scaler=scaler,
                                factor_alpha=factor_alpha, ema=ema, fgm=fgm)
        print(f"  epoch {epoch}: loss {loss:.4f}  ({time.time()-t0:.0f}s elapsed)")

    # eval/inference use the EMA-smoothed weights when enabled; training itself never
    # resumes from the shadow (apply_shadow/restore always bracket eval-only code).
    if ema is not None:
        ema.apply_shadow(model)

    # --- evaluate on this fold's held-out validation posts (default 0.5/0.5/1 decode --
    #     the pooled, globally-tuned decode happens once after the loop, below) ---
    va_posts = X_text[va_idx].tolist()
    risk_probs, factor_probs, evidence_probs, offsets_list, risk_cls_probs, risk_coral_probs = predict_probs(
        model, va_posts, tokenizer, device, MAX_LENGTH, return_risk_components=True)
    metrics = evaluate_fold(risk_probs, factor_probs, evidence_probs, offsets_list, va_posts,
                             y_risk[va_idx], [y_factors[i] for i in va_idx], [y_evidence[i] for i in va_idx])
    print(f"  fold {fold} val metrics (default thresholds): {metrics}")
    fold_results.append(metrics)

    # stash OOF probabilities (not yet thresholded/blended) for every row this fold
    # held out -- risk/factors/evidence all get decoded once, globally, after the loop.
    for j, orig_i in enumerate(va_idx):
        oof_risk_true[orig_i] = y_risk[orig_i]
        oof_risk_cls_probs[orig_i] = risk_cls_probs[j]
        oof_risk_coral_probs[orig_i] = risk_coral_probs[j]
        oof_factors_true[orig_i] = y_factors[orig_i]
        oof_factor_probs[orig_i] = factor_probs[j]
        oof_evidence_probs[orig_i] = evidence_probs[j]
        oof_evidence_offsets[orig_i] = offsets_list[j]

    lb_posts = leaderboard_df["post_clean"].tolist()

    # --- classical TF-IDF + logistic-regression factors model, same fold split ---
    # Fit on the SAME oversampled training posts the neural model just trained on.
    # Blended with the neural probabilities post-loop (see after the loop), not used alone.
    classical_vectorizer, classical_models = fit_classical_factors_model(
        X_text[tr_idx_oversampled].tolist(), [y_factors[i] for i in tr_idx_oversampled], len(FACTOR_TAXONOMY))
    classical_va_probs = predict_classical_factors(classical_vectorizer, classical_models, va_posts, len(FACTOR_TAXONOMY))
    classical_lb_probs = predict_classical_factors(classical_vectorizer, classical_models, lb_posts, len(FACTOR_TAXONOMY))
    for j, orig_i in enumerate(va_idx):
        oof_classical_factor_probs[orig_i] = classical_va_probs[j]
    lb_classical_factor_probs_sum += classical_lb_probs

    # --- predict on the leaderboard set now, while this fold's model is loaded, and
    #     accumulate into a running average (this IS the ensemble) ---
    _, lfp, lep, loff, lrcls, lrcoral = predict_probs(
        model, lb_posts, tokenizer, device, MAX_LENGTH, return_risk_components=True)
    lb_risk_cls_probs_sum += lrcls
    lb_risk_coral_probs_sum += lrcoral
    lb_factor_probs_sum += lfp
    if lb_evidence_probs_sum is None:
        lb_evidence_probs_sum = [np.array(p, dtype=float) for p in lep]
        lb_offsets_ref = loff
    else:
        for i in range(len(lep)):
            lb_evidence_probs_sum[i] += np.array(lep[i], dtype=float)

    if ema is not None:
        ema.restore(model)  # hygiene -- model is about to be discarded anyway

    del model, encoder, optimizer, scheduler
    if device.type == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(fold_results)
print("\n" + "="*60)
print(f"{N_FOLDS}-FOLD CV SUMMARY (default 0.5/0.5/1 thresholds)")
print("="*60)
for col in ["risk_f1", "evidence_f1", "factors_f1", "composite"]:
    print(f"{col:26s}: {results_df[col].mean():.4f} +/- {results_df[col].std():.4f}")

# ---- Pooled out-of-fold calibration (replaces the old per-fold dev-split tuning) ----
# Every prediction pooled here came from a model that never trained on that row (proper
# OOF), but now spans the FULL 1635-row train set instead of a ~245-row per-fold dev
# slice -- e.g. a category at ~1% prevalence goes from single-digit dev positives (where
# an F1-argmax search is really just fitting noise) to ~16 pooled positives, enough to
# search a threshold against with some real confidence. This does look at OOF labels to
# pick thresholds/hparams, so treat the resulting F1 below as a standard, mildly
# optimistic-but-far-more-trustworthy CV estimate -- not as clean as a fully untouched
# held-out set, but a real improvement over averaging five noisy per-fold thresholds.
# ---- Risk ensemble weight (replaces the fixed 50/50 CLS/CORAL blend) ----
# Never tuned before this round -- same pooled-OOF philosophy as the factor/evidence
# calibration above, just applied to the one risk-side knob that was still hardcoded.
best_risk_w, best_risk_f1 = 0.5, -1.0
default_risk_f1 = None
for w in np.linspace(0.0, 1.0, 11):
    blended = w * oof_risk_cls_probs + (1 - w) * oof_risk_coral_probs
    pred = [RISK_LEVELS[i] for i in blended.argmax(axis=1)]
    f1 = weighted_f1_risk(oof_risk_true, pred)
    if abs(w - 0.5) < 1e-9:
        default_risk_f1 = f1
    if f1 > best_risk_f1:
        best_risk_f1, best_risk_w = f1, w
print(f"risk weighted F1, OOF, default w=0.5 (CLS/CORAL): {default_risk_f1:.4f}")
print(f"risk weighted F1, OOF, pooled-tuned w={best_risk_w:.1f}          : {best_risk_f1:.4f}")
oof_risk_blended = best_risk_w * oof_risk_cls_probs + (1 - best_risk_w) * oof_risk_coral_probs
oof_risk_pred = [RISK_LEVELS[i] for i in oof_risk_blended.argmax(axis=1)]

oof_factors_true_mat = np.stack([factors_multihot(oof_factors_true[i]) for i in range(len(train_df))])

# ---- Neural/classical ensemble weight for factors (new this round) ----
# alpha=1.0 is pure neural (every previous round's approach); the search below always
# includes it as a candidate, so this can only match or beat pure-neural on the pooled-OOF
# metric, never silently do worse there. As always, the real test is the submission --
# OOF and real leaderboard factors scores have disagreed before this session (see the
# F-beta gate finding).
best_alpha, best_alpha_f1, global_factor_thresholds = 1.0, -1.0, None
for alpha in np.linspace(0.0, 1.0, 11):
    blended_probs = alpha * oof_factor_probs + (1 - alpha) * oof_classical_factor_probs
    cand_thresholds = tune_factor_thresholds(
        blended_probs, oof_factors_true_mat, len(FACTOR_TAXONOMY),
        min_support=MIN_SUPPORT_FOR_THRESHOLD_TUNING, fbeta=FACTOR_THRESHOLD_FBETA,
        overpredict_ratio=FACTOR_OVERPREDICT_RATIO, category_names=FACTOR_TAXONOMY, verbose=False)
    cand_pred = [[FACTOR_TAXONOMY[c] for c in range(len(FACTOR_TAXONOMY)) if blended_probs[i, c] > cand_thresholds[c]]
                 for i in range(len(train_df))]
    cand_f1 = macro_f1_factors(oof_factors_true, cand_pred)
    if cand_f1 > best_alpha_f1:
        best_alpha, best_alpha_f1, global_factor_thresholds = alpha, cand_f1, cand_thresholds
print(f"factors neural/classical blend: best alpha (neural weight) = {best_alpha:.1f}, "
      f"OOF macro F1 = {best_alpha_f1:.4f} (alpha=1.0 = pure neural, i.e. every prior round)")

oof_blended_factor_probs = best_alpha * oof_factor_probs + (1 - best_alpha) * oof_classical_factor_probs
default_factor_pred = [[FACTOR_TAXONOMY[c] for c in range(len(FACTOR_TAXONOMY)) if oof_factor_probs[i, c] > 0.5]
                        for i in range(len(train_df))]
tuned_factor_pred = [[FACTOR_TAXONOMY[c] for c in range(len(FACTOR_TAXONOMY)) if oof_blended_factor_probs[i, c] > global_factor_thresholds[c]]
                     for i in range(len(train_df))]
print(f"\nfactors macro F1, OOF, default 0.5 threshold (pure neural)   : {macro_f1_factors(oof_factors_true, default_factor_pred):.4f}")
print(f"factors macro F1, OOF, pooled-tuned threshold (best blend)   : {macro_f1_factors(oof_factors_true, tuned_factor_pred):.4f}")
oof_factors_pred = tuned_factor_pred  # what the error-analysis cell below reads

# ---- Optional third signal: LLM zero-shot factor classification ----
# best_beta=1.0 means "LLM signal not used" (either the cell above failed, or the search below
# simply never found a beta<1.0 that beat the neural+classical blend on pooled OOF) -- in that
# case global_factor_thresholds/oof_factors_pred are left exactly as the first-stage search set
# them, unchanged. Always defined by the time the final-ensemble cell runs, so that cell never
# needs to check whether this block ran.
best_beta = 1.0
if llm_factor_probs_train is not None:
    best_beta, best_beta_f1 = 1.0, best_alpha_f1
    for beta in np.linspace(0.0, 1.0, 11):
        cand_blend = beta * oof_blended_factor_probs + (1 - beta) * llm_factor_probs_train
        cand_thresholds = tune_factor_thresholds(
            cand_blend, oof_factors_true_mat, len(FACTOR_TAXONOMY),
            min_support=MIN_SUPPORT_FOR_THRESHOLD_TUNING, fbeta=FACTOR_THRESHOLD_FBETA,
            overpredict_ratio=FACTOR_OVERPREDICT_RATIO, category_names=FACTOR_TAXONOMY, verbose=False)
        cand_pred = [[FACTOR_TAXONOMY[c] for c in range(len(FACTOR_TAXONOMY)) if cand_blend[i, c] > cand_thresholds[c]]
                     for i in range(len(train_df))]
        cand_f1 = macro_f1_factors(oof_factors_true, cand_pred)
        if cand_f1 > best_beta_f1:
            best_beta, best_beta_f1, global_factor_thresholds = beta, cand_f1, cand_thresholds
    print(f"factors +LLM blend: best beta (neural+classical weight) = {best_beta:.1f}, "
          f"OOF macro F1 = {best_beta_f1:.4f} (beta=1.0 = LLM signal not used)")
    oof_final_factor_probs = best_beta * oof_blended_factor_probs + (1 - best_beta) * llm_factor_probs_train
    oof_factors_pred = [[FACTOR_TAXONOMY[c] for c in range(len(FACTOR_TAXONOMY)) if oof_final_factor_probs[i, c] > global_factor_thresholds[c]]
                         for i in range(len(train_df))]
    print(f"factors macro F1, OOF, pooled-tuned threshold (neural+classical+LLM): {macro_f1_factors(oof_factors_true, oof_factors_pred):.4f}")
else:
    print("LLM factor signal not available this run -- using neural+classical blend only.")

oof_posts = train_df["post_clean"].tolist()
default_evidence_pred = [
    extract_evidence_spans((np.array(p) > 0.5).astype(int).tolist(), o, t, merge_gap=1)
    for p, o, t in zip(oof_evidence_probs, oof_evidence_offsets, oof_posts)
]
default_ev_f1 = phrase_f1_corpus(default_evidence_pred, y_evidence)
ev_f1, ev_threshold, ev_merge_gap = tune_evidence_hparams(
    oof_evidence_probs, oof_evidence_offsets, oof_posts, y_evidence)
print(f"\nevidence phrase F1, OOF, default (threshold=0.5, merge_gap=1)   : {default_ev_f1:.4f}")
print(f"evidence phrase F1, OOF, pooled-tuned (threshold={ev_threshold}, merge_gap={ev_merge_gap}): {ev_f1:.4f}")

Averages each fold's predicted probabilities (not hard labels) across all `N_FOLDS` models — generally more robust than any single fold, and is the same idea as the "ensemble inference" from the earlier notebook work on this project. Decoded using the pooled-out-of-fold-tuned factor thresholds and evidence (threshold, merge_gap) from the previous cell, rather than the old fixed 0.5/0.5/1 defaults.

In [ ]:
lb_risk_cls_probs = lb_risk_cls_probs_sum / N_FOLDS
lb_risk_coral_probs = lb_risk_coral_probs_sum / N_FOLDS
lb_risk_probs = best_risk_w * lb_risk_cls_probs + (1 - best_risk_w) * lb_risk_coral_probs
lb_factor_probs_neural = lb_factor_probs_sum / N_FOLDS
lb_classical_factor_probs = lb_classical_factor_probs_sum / N_FOLDS
lb_factor_probs = best_alpha * lb_factor_probs_neural + (1 - best_alpha) * lb_classical_factor_probs
if llm_factor_probs_lb is not None:
    # best_beta==1.0 (LLM unused or unavailable) makes this a no-op, same value as above
    lb_factor_probs = best_beta * lb_factor_probs + (1 - best_beta) * llm_factor_probs_lb
lb_evidence_probs = [p / N_FOLDS for p in lb_evidence_probs_sum]

risk_pred, factor_pred, evidence_pred = decode_predictions(
    lb_risk_probs, lb_factor_probs, lb_evidence_probs, lb_offsets_ref,
    leaderboard_df["post_clean"].tolist(),
    factor_threshold=global_factor_thresholds,
    evidence_threshold=ev_threshold, evidence_merge_gap=ev_merge_gap)

submission = pd.DataFrame({
    "row_id": leaderboard_df["row_id"].values,
    "risk_level": risk_pred,
    "evidence": ["; ".join(s) for s in evidence_pred],
    "factors": [list(f) for f in factor_pred],
})
out_path = f"{TEAM_NAME}.csv"
submission.to_csv(out_path, index=False)
print(f"wrote {len(submission)} rows -> {out_path}")
print(submission["risk_level"].value_counts())
print("\nmean predicted factors/row:", submission["factors"].apply(len).mean(),
      " (training set's true mean is ~3.5 -- if this is far above that, the calibration")
print("  fix didn't fully land; if it's close, the threshold tuning is working as intended)")
submission.head()

## Error analysis: confusion matrix + per-category factor errors
Uses the **out-of-fold** predictions collected during CV — every training row was scored by whichever fold held it out, so this covers all of `train_df`, not just one fold's slice. Useful for spotting *which* risk levels get confused with each other and *which* factor categories are still weak after the threshold fix.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

valid_oof = [i for i in range(len(train_df)) if oof_risk_pred[i] is not None]
y_true_oof = [oof_risk_true[i] for i in valid_oof]
y_pred_oof = [oof_risk_pred[i] for i in valid_oof]

print("RISK LEVEL -- out-of-fold confusion matrix (rows=true, cols=predicted)")
cm = confusion_matrix(y_true_oof, y_pred_oof, labels=RISK_LEVELS)
print(pd.DataFrame(cm, index=RISK_LEVELS, columns=RISK_LEVELS))
print()
print(classification_report(y_true_oof, y_pred_oof, labels=RISK_LEVELS, zero_division=0))

print("\nFACTORS -- per-category out-of-fold precision/recall/F1 (tuned thresholds)")
mlb_report = MultiLabelBinarizer(classes=FACTOR_TAXONOMY)
Yt = mlb_report.fit_transform([oof_factors_true[i] for i in valid_oof])
Yp = mlb_report.transform([oof_factors_pred[i] for i in valid_oof])
per_cat = pd.DataFrame({
    "category": FACTOR_TAXONOMY,
    "true_prevalence": Yt.mean(axis=0),
    "predicted_rate": Yp.mean(axis=0),
    "f1": f1_score(Yt, Yp, average=None, zero_division=0),
}).sort_values("f1")
print(per_cat.to_string(index=False))
print("\nWorst 5 categories -- if predicted_rate is still >>2x true_prevalence for any of")
print("these, that category specifically may need its own attention (more epochs won't")
print("fix a category with too few positive examples to learn from at all).")